# 💼 Job Hunt Crew

## What We're Building

A 4-agent job application assistant:

| Agent | Role |
|-------|------|
| **JD Analyzer** | Parse job descriptions and extract key requirements |
| **Resume Tailor** | Customize the resume to match the JD |
| **Cover Letter Writer** | Write a tailored cover letter |
| **Interview Prep Coach** | Generate likely interview questions and answers |

## Key CrewAI Concept: `allow_delegation`

The interview prep coach can delegate research back to the JD analyzer
if it needs more context about specific technical requirements.

## Stack
- **CrewAI** — multi-agent pipeline
- **Ollama** — local LLM

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
print(f"LLM ready: {llm.model}")

## Step 2 — Job Description & Resume

In [ ]:
job_description = """
Role: Senior Machine Learning Engineer
Company: HealthAI Inc. (Series B, $50M raised)
Location: Remote (US)
Salary: $180,000 - $220,000 + equity

About us: HealthAI builds AI-powered diagnostic tools for hospitals.
We process millions of medical images daily using deep learning.

Requirements:
- 5+ years ML engineering experience
- Strong Python (PyTorch preferred)
- Experience with computer vision / image classification
- MLOps: model deployment, monitoring, CI/CD for ML
- Experience with cloud (AWS or GCP)
- Healthcare/regulated industry experience is a plus
- Published papers or patents is a bonus

Nice-to-have:
- Experience with HIPAA compliance
- Knowledge of medical imaging (DICOM, radiology)
- Team lead experience

What you'll do:
- Lead development of image classification models for radiology
- Build and maintain ML pipelines (training, evaluation, deployment)
- Collaborate with clinicians to validate model performance
- Mentor junior engineers
"""

candidate_resume = """
Name: Alex Chen
Current Role: ML Engineer at TechCorp (3 years)
Previous: Data Scientist at RetailAI (2 years)
Education: MS Computer Science, Stanford University

Skills:
- Python, PyTorch, TensorFlow, scikit-learn
- Computer Vision: image classification, object detection, segmentation
- MLOps: MLflow, Docker, Kubernetes, AWS SageMaker
- Cloud: AWS (3 years), GCP (1 year)
- Languages: Python, SQL, Bash

Experience highlights:
- Built a product defect detection system (98.5% accuracy, $2M/year savings)
- Deployed 15+ models to production using MLflow + K8s
- Led a team of 3 junior engineers
- Published 2 papers at CVPR on efficient architecture search

Missing from resume:
- No healthcare industry experience
- No HIPAA experience
- No medical imaging (DICOM) experience
"""

print("JD and resume loaded")

## Step 3 — Create Agents

In [ ]:
jd_analyzer = Agent(
    role="Job Description Analyst",
    goal="Parse job descriptions to extract requirements, priorities, and hidden signals",
    backstory=(
        "You are a career coach who has reviewed 5,000+ job descriptions. "
        "You can read between the lines — you know what's truly required vs. "
        "wish-list items, and you spot company culture signals from the language."
    ),
    llm=llm,
    verbose=True,
)

resume_tailor = Agent(
    role="Resume Optimization Specialist",
    goal="Tailor the candidate's resume to maximize match with the job description",
    backstory=(
        "You are a professional resume writer who has helped 500+ engineers land "
        "FAANG and Series B startup roles. You know ATS systems, keyword matching, "
        "and how to frame experience to match job requirements. You never fabricate — "
        "you reframe existing experience truthfully."
    ),
    llm=llm,
    verbose=True,
)

cover_letter_writer = Agent(
    role="Cover Letter Specialist",
    goal="Write a compelling cover letter that addresses gaps and highlights fit",
    backstory=(
        "You write cover letters that hiring managers actually read. Your formula: "
        "hook with a relevant achievement, connect your experience to their needs, "
        "address any gaps proactively, and close with enthusiasm. Never generic."
    ),
    llm=llm,
    verbose=True,
)

interview_coach = Agent(
    role="Interview Preparation Coach",
    goal="Prepare the candidate with likely interview questions and strong answers",
    backstory=(
        "You are a senior engineering manager who has conducted 500+ interviews. "
        "You know exactly what interviewers look for at each level. You prep candidates "
        "with STAR-format answers and help them handle tricky questions about gaps."
    ),
    llm=llm,
    allow_delegation=True,  # Can ask JD analyzer for more details
    verbose=True,
)

print("4 agents created")

## Step 4 — Create Tasks

In [ ]:
analyze_task = Task(
    description=f"""Analyze this job description:

{job_description}

Extract:
1. MUST-HAVE requirements (true dealbreakers)
2. NICE-TO-HAVE requirements (differentiators but not required)
3. HIDDEN SIGNALS (what the JD implies about culture, team, urgency)
4. KEY THEMES (what the company values most)
5. RED FLAGS or concerns for candidates
6. Ideal candidate persona""",
    expected_output="Structured JD analysis with priorities and signals.",
    agent=jd_analyzer,
)

resume_task = Task(
    description=f"""Tailor this resume to match the job description analysis:

CANDIDATE RESUME:
{candidate_resume}

Instructions:
1. Rewrite the summary/objective to target this specific role
2. Reorder and reframe bullet points to emphasize relevant skills
3. Add keywords from the JD that the candidate genuinely has
4. Highlight transferable skills for areas where there's no direct experience
5. Suggest what to remove or de-emphasize
6. DO NOT fabricate experience""",
    expected_output="A tailored resume with keyword optimization.",
    agent=resume_tailor,
)

cover_letter_task = Task(
    description=f"""Write a cover letter for this job application.

JOB: Senior ML Engineer at HealthAI Inc.
CANDIDATE: Alex Chen

The cover letter should:
1. Open with a specific, relevant achievement (not "I am writing to apply")
2. Show understanding of HealthAI's mission and challenges
3. Connect 2-3 key experiences directly to their requirements
4. Address the healthcare experience gap proactively (show transferability)
5. Close with genuine enthusiasm and a specific next step
6. Be 3-4 paragraphs, under 400 words""",
    expected_output="A personalized cover letter under 400 words.",
    agent=cover_letter_writer,
)

interview_task = Task(
    description=f"""Prepare interview questions and answers for this candidate.

Generate:
1. 5 TECHNICAL questions they'll likely face (with model answers)
   - Focus on PyTorch, computer vision, MLOps
2. 3 BEHAVIORAL questions using STAR format
   - Include 'Tell me about a time you led a team' and 'Tell me about a failure'
3. 2 TRICKY QUESTIONS about gaps
   - 'Why healthcare with no healthcare experience?'
   - 'What do you know about HIPAA/medical imaging?'
4. 3 QUESTIONS THE CANDIDATE SHOULD ASK the interviewer
5. Common pitfalls to avoid""",
    expected_output="Interview prep guide with questions, answers, and tips.",
    agent=interview_coach,
)

print("4 tasks created")

## Step 5 — Run the Crew

In [ ]:
job_crew = Crew(
    agents=[jd_analyzer, resume_tailor, cover_letter_writer, interview_coach],
    tasks=[analyze_task, resume_task, cover_letter_task, interview_task],
    process=Process.sequential,
    verbose=True,
)

print("Job hunt crew assembled!")
result = job_crew.kickoff()
print("\n✅ Job application package complete!")

In [ ]:
print("🎤 INTERVIEW PREP GUIDE:")
print("=" * 60)
print(result.raw)

# Show all outputs
labels = ["JD Analysis", "Tailored Resume", "Cover Letter", "Interview Prep"]
for label, task_out in zip(labels, result.tasks_output):
    print(f"\n{'━'*60}")
    print(f"📌 {label}")
    print(f"{'━'*60}")
    print(task_out.raw[:500])
    if len(task_out.raw) > 500:
        print("...")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **allow_delegation** | Interview coach can delegate to JD analyzer for details |
| **Gap addressing** | Cover letter proactively addresses missing healthcare experience |
| **STAR format** | Interview prep uses Situation-Task-Action-Result structure |
| **Truthful tailoring** | Resume tailor reframes but never fabricates experience |

## 🔧 Extensions

- **Job board scraping**: Use tools to pull real job descriptions
- **Multiple applications**: Use `kickoff_for_each()` for batch applications
- **Mock interview**: Add a simulated interviewer agent for practice
- **ATS scoring**: Add a task that scores the resume against ATS algorithms